# NB6B — MIN2 dùng frozen FashionCLIP cache V2

Notebook này thay thế flow auto-embed của NB6 cũ.

Contract:

```text
Compatibility scorer: 2–8 items
LOO diagnosis: original outfit >= 3 items
Embedding: reuse frozen FashionCLIP cache referenced by artifacts/data_v2_reference.json
```

Nguyên tắc quan trọng:

1. `artifacts/data_v2_reference.json` là source of truth cho external storage, filename và SHA-256.
2. Notebook tìm + tải đúng `fashionclip_item_embeddings.pt` và `embedding_manifest_v1.json` từ Google Drive reference.
3. SHA-256 phải match frozen V2 reference trước khi dùng.
4. Không chạy FashionCLIP lại.
5. Nếu frozen cache không cover các item mới xuất hiện ở MIN2, notebook dừng và report missing IDs; không tự tạo embedding mới.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

EXPECTED_BRANCH = "exp/min2-scorer-loo3"
REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"

def is_project_repo(path: Path) -> bool:
    path = path.expanduser().resolve()
    return (
        (path / ".git").exists()
        and (path / "src/data/min2_experiment.py").is_file()
        and (path / "artifacts/data_v2_reference.json").is_file()
    )

candidates = []
if os.environ.get("FASHION_PROJECT_ROOT"):
    candidates.append(Path(os.environ["FASHION_PROJECT_ROOT"]))

cwd = Path.cwd().resolve()
candidates.extend([cwd, *cwd.parents])
if Path("/content").exists():
    candidates.append(Path("/content/opisoverated"))
candidates.append(Path.home() / "opisoverated")

ROOT = next((p.expanduser().resolve() for p in candidates if is_project_repo(p)), None)

if ROOT is None:
    clone_parent = Path("/content") if Path("/content").exists() else Path.home()
    ROOT = clone_parent / "opisoverated"
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(
            f"{ROOT} exists but is not a usable checkout. "
            "Remove/rename it or set FASHION_PROJECT_ROOT."
        )
    subprocess.run(
        ["git", "clone", "--branch", EXPECTED_BRANCH, "--single-branch", REPO_URL, str(ROOT)],
        check=True,
    )

branch = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "--abbrev-ref", "HEAD"], text=True
).strip()

if branch != EXPECTED_BRANCH:
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", EXPECTED_BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", EXPECTED_BRANCH], check=True)

subprocess.run(
    ["git", "-C", str(ROOT), "pull", "--ff-only", "origin", EXPECTED_BRANCH],
    check=True,
)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("repo root :", ROOT)
print("branch    :", subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "--abbrev-ref", "HEAD"], text=True
).strip())
print("commit    :", subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True
).strip())


## 1. Đọc frozen V2 artifact reference

Cell này không đoán đường dẫn embedding. Nó đọc trực tiếp:

`artifacts/data_v2_reference.json`


In [ ]:
REFERENCE_PATH = ROOT / "artifacts/data_v2_reference.json"

with REFERENCE_PATH.open("r", encoding="utf-8") as f:
    DATA_REF = json.load(f)

storage = DATA_REF["storage"]
layout = DATA_REF["external_layout"]
freeze = DATA_REF["freeze_fields_after_rebuild"]

assert DATA_REF["status"] == "READY_TO_TRAIN"
assert DATA_REF["embedding_version"] == "fashionclip-512-l2-v1"
assert storage["provider"] == "google_drive"

DRIVE_FOLDER_URL = storage["folder_url"]
CACHE_FILENAME = layout["embedding_cache"]
MANIFEST_FILENAME = layout["embedding_manifest"]

EXPECTED_CACHE_SHA256 = freeze["embedding_cache_sha256"]
EXPECTED_MANIFEST_SHA256 = freeze["embedding_manifest_sha256"]

print("dataset version        :", DATA_REF["dataset_version"])
print("embedding version      :", DATA_REF["embedding_version"])
print("Drive reference folder :", DRIVE_FOLDER_URL)
print("cache filename         :", CACHE_FILENAME)
print("manifest filename      :", MANIFEST_FILENAME)
print("cache sha256           :", EXPECTED_CACHE_SHA256)
print("manifest sha256        :", EXPECTED_MANIFEST_SHA256)
print("core7 external dir     :", layout["core7_dir"])
print("scorer-ready ext dir   :", layout["scorer_ready_dir"])


## 2. Runtime paths của MIN2

Frozen cache/manifest sẽ được download/copy vào path mà MIN2 validator đang dùng.

Việc này **không thay đổi nội dung frozen cache**; chỉ materialize external artifact vào runtime Colab.


In [ ]:
from src.data.runtime_paths import load_runtime_paths
from src.data.min2_experiment import (
    EXPERIMENT_DATASET_VERSION,
    EXPERIMENT_TAG,
    MIN_SCORER_ITEMS,
    MAX_SCORER_ITEMS,
    LOO_MIN_ORIGINAL_ITEMS,
    prepare_min2_positives,
    validate_min2_embeddings,
    build_min2_scorer_dataset,
    scorer_ready_path,
)
from src.scorer.min2_experiment import (
    load_config,
    build_train_valid_loaders_min2,
    build_min2_datasets,
    build_min2_provenance,
    fit_min2_scorer,
    collate_min2_scorer_batch,
)
from src.scorer.model import TypeAwarePairwiseScorer
from src.scorer.train import seed_everything
from src.scorer.evaluate import evaluate_model
from src.scorer import checkpoint as checkpoint_utils
from src.scorer.dataset import read_jsonl
from src.diagnosis.evaluate_loo import evaluate_loo_dataset

PATHS_CONFIG = ROOT / "configs/data_paths.min2_experiment.json"
SCORER_CONFIG = ROOT / "configs/scorer_type_aware_pairwise_min2_experiment.yaml"
MAPPING = ROOT / "configs/category_mapping_core7_v2.json"

paths = load_runtime_paths(repo_root=ROOT, config_path=PATHS_CONFIG)
config = load_config(SCORER_CONFIG)

print("MIN2 dataset version:", EXPERIMENT_DATASET_VERSION)
print("scorer min/max      :", MIN_SCORER_ITEMS, MAX_SCORER_ITEMS)
print("LOO min original    :", LOO_MIN_ORIGINAL_ITEMS)
print("embedding cache dst :", paths.embedding_cache)
print("embedding manifest  :", paths.embedding_manifest)
print("core7 MIN2 dir      :", paths.core7_dir)
print("scorer-ready MIN2   :", paths.scorer_ready_dir)


## 3. Lấy đúng frozen FashionCLIP artifacts từ Google Drive

Notebook authenticate Google Drive API, bắt đầu từ `storage.folder_url` trong reference JSON, duyệt recursive và tìm đúng hai filename.

Sau khi download:
- SHA-256 của `.pt` phải bằng `embedding_cache_sha256`;
- SHA-256 của manifest phải bằng `embedding_manifest_sha256`.

Nếu không match, notebook hard-fail.


In [ ]:
import hashlib
import re

try:
    from google.colab import auth
except ModuleNotFoundError as e:
    raise RuntimeError(
        "Cell này được thiết kế cho Google Colab vì frozen artifacts nằm trên Google Drive."
    ) from e

auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

creds, _ = google.auth.default()
drive_service = build("drive", "v3", credentials=creds, cache_discovery=False)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def folder_id_from_url(url: str) -> str:
    match = re.search(r"/folders/([A-Za-z0-9_-]+)", url)
    if not match:
        raise ValueError(f"Cannot parse Google Drive folder id from: {url}")
    return match.group(1)

def list_children(folder_id: str):
    page_token = None
    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id,name,mimeType,size,parents)",
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        yield from response.get("files", [])
        page_token = response.get("nextPageToken")
        if not page_token:
            break

def find_named_files_recursive(root_folder_id: str, target_names: set[str]):
    found = {}
    queue = [root_folder_id]
    visited = set()

    while queue and set(found) != target_names:
        folder_id = queue.pop(0)
        if folder_id in visited:
            continue
        visited.add(folder_id)

        for item in list_children(folder_id):
            name = item["name"]
            mime = item["mimeType"]
            if name in target_names and name not in found:
                found[name] = item
                print("found:", name, "id=", item["id"], "size=", item.get("size"))
            if mime == "application/vnd.google-apps.folder":
                queue.append(item["id"])

    return found

def download_drive_file(file_id: str, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp = destination.with_suffix(destination.suffix + ".partial")

    request = drive_service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with temp.open("wb") as f:
        downloader = MediaIoBaseDownload(f, request, chunksize=32 * 1024 * 1024)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"  {destination.name}: {status.progress() * 100:5.1f}%")

    temp.replace(destination)

ROOT_FOLDER_ID = folder_id_from_url(DRIVE_FOLDER_URL)
targets = {CACHE_FILENAME, MANIFEST_FILENAME}
found = find_named_files_recursive(ROOT_FOLDER_ID, targets)

missing_names = sorted(targets - set(found))
if missing_names:
    raise FileNotFoundError(
        "Không tìm thấy artifact được reference bên dưới storage.folder_url: "
        + repr(missing_names)
        + ". Không chạy FashionCLIP lại. Hãy kiểm tra quyền Drive/reference folder."
    )

expected = {
    CACHE_FILENAME: (paths.embedding_cache, EXPECTED_CACHE_SHA256),
    MANIFEST_FILENAME: (paths.embedding_manifest, EXPECTED_MANIFEST_SHA256),
}

for name, (destination, expected_sha) in expected.items():
    if destination.is_file() and sha256_file(destination) == expected_sha:
        print("reuse local exact artifact:", destination)
        continue

    print("downloading exact frozen artifact:", name)
    download_drive_file(found[name]["id"], destination)

    actual_sha = sha256_file(destination)
    print("  sha256:", actual_sha)
    if actual_sha != expected_sha:
        destination.unlink(missing_ok=True)
        raise ValueError(
            f"{name} SHA-256 mismatch. expected={expected_sha}, actual={actual_sha}"
        )

print("Frozen cache + manifest match data_v2_reference.json.")


## 4. Inspect manifest + cache contract

Đây là validation của artifact đã freeze, không phải regenerate embedding.


In [ ]:
import torch
from src.data.validate_core7_embeddings import (
    read_json,
    load_embedding_cache,
    inspect_embedding_cache,
    inspect_embedding_manifest,
)

manifest = read_json(paths.embedding_manifest)
cache = load_embedding_cache(paths.embedding_cache)
cache_report, usable_item_ids = inspect_embedding_cache(cache)
manifest_report = inspect_embedding_manifest(
    manifest,
    cache_report=cache_report,
    cache_sha256=sha256_file(paths.embedding_cache),
)

print("manifest:")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\ncache checks:")
print(json.dumps(cache_report, ensure_ascii=False, indent=2))
print("\nmanifest checks:")
print(json.dumps(manifest_report, ensure_ascii=False, indent=2))

assert cache_report["pass"], "Frozen embedding cache schema/content failed."
assert manifest_report["pass"], "Frozen embedding manifest failed."
assert manifest["embedding_version"] == DATA_REF["embedding_version"]
assert manifest["cache_sha256"] == EXPECTED_CACHE_SHA256


## 5. Regenerate MIN2 positives

Chỉ data filtering được chạy lại với `min_items=2`.

Không gọi `ensure_min2_embedding_cache` và không gọi FashionCLIP.


In [ ]:
DEBUG_LIMIT = None  # None = full train/valid/test

prepare_report = prepare_min2_positives(
    paths,
    mapping_path=MAPPING,
    debug_limit=DEBUG_LIMIT,
)

for split, report in prepare_report["splits"].items():
    print(
        split,
        "kept=", report["outfits"]["outfits_kept"],
        "lengths=", report["outfits"]["outfit_length_distribution_after"],
    )


## 6. Validate frozen cache coverage trên MIN2

Đây là gate quan trọng nhất của experiment.

Nếu `pass=False`, frozen V2 cache không cover một số item mới được MIN2 đưa vào. Notebook sẽ report số lượng + ví dụ item ID và dừng. **Không tự re-embed.**


In [ ]:
embedding_report = validate_min2_embeddings(paths, mapping_path=MAPPING)

for split, report in embedding_report["splits"].items():
    print(
        split,
        "coverage=", f'{report["embedding_coverage"]:.6%}',
        "missing=", report["missing_or_invalid_embedding_count"],
    )
    if report["missing_or_invalid_embedding_count"]:
        print("  examples:", report["missing_or_invalid_embedding_examples"][:20])

if not embedding_report["pass"]:
    raise RuntimeError(
        "Frozen FashionCLIP V2 cache không đạt exact MIN2 coverage. "
        "Dừng ở đây để quyết định explicit embedding-version extension; "
        "không tự chạy FashionCLIP lại."
    )

print("Embedding validation PASS: frozen cache covers MIN2 exactly.")


## 7. Build scorer-ready MIN2 dataset


In [ ]:
dataset_report = build_min2_scorer_dataset(
    paths,
    mapping_path=MAPPING,
    overwrite=True,
)

print(json.dumps(dataset_report, ensure_ascii=False, indent=2))
assert dataset_report["status"] == "READY_TO_TRAIN"


In [ ]:
from collections import Counter

for split in ("train", "valid", "test"):
    rows = read_jsonl(scorer_ready_path(paths.scorer_ready_dir, split))
    lengths = Counter(len(row["items"]) for row in rows)
    labels = Counter(int(row["label"]) for row in rows)
    print(
        split,
        "samples=", len(rows),
        "labels=", dict(labels),
        "lengths=", dict(sorted(lengths.items())),
    )
    assert lengths.get(2, 0) > 0, f"{split}: expected MIN2 samples are absent"


## 8. Train Type-aware Pairwise scorer từ scratch

Architecture/V5 optimization giữ nguyên. Chỉ data contract cho scorer đổi thành 2–8.


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = (
    paths.artifact_root
    / "checkpoints"
    / "type_aware_pairwise_v1"
    / "min2_exp_v1_seed42"
)

seed_everything(int(config["training"]["seed"]))
model = TypeAwarePairwiseScorer.from_config(config)
loaders = build_train_valid_loaders_min2(paths, config, num_workers=0)
provenance = build_min2_provenance(paths, ROOT)

train_result = fit_min2_scorer(
    model,
    loaders["train_loader"],
    loaders["valid_loader"],
    config=config,
    checkpoint_dir=CHECKPOINT_DIR,
    provenance=provenance,
    device=DEVICE,
)

print("device             :", DEVICE)
print("best epoch         :", train_result["best_epoch"])
print("best valid ROC-AUC :", train_result["best_valid_roc_auc"])
print("best checkpoint    :", train_result["best_checkpoint"])


## 9. Load best checkpoint + scorer evaluation


In [ ]:
from torch.utils.data import DataLoader
from functools import partial

best_model = TypeAwarePairwiseScorer.from_config(config)
ckpt = checkpoint_utils.load_checkpoint(
    train_result["best_checkpoint"],
    model=best_model,
    map_location="cpu",
)
best_model.to(DEVICE).eval()

EVALUATE_TEST = True
eval_splits = ["valid"] + (["test"] if EVALUATE_TEST else [])

eval_datasets, _ = build_min2_datasets(paths, splits=tuple(eval_splits))

scorer_metrics = {}
for split in eval_splits:
    loader = DataLoader(
        eval_datasets[split],
        batch_size=int(config["training"]["batch_size"]),
        shuffle=False,
        collate_fn=partial(
            collate_min2_scorer_batch,
            max_items=MAX_SCORER_ITEMS,
        ),
    )
    result = evaluate_model(best_model, loader, device=DEVICE)
    scorer_metrics[split] = result["metrics"]
    print("\nSCORER", split)
    print(json.dumps(result["metrics"], ensure_ascii=False, indent=2))


## 10. LOO diagnosis evaluation

LOO chỉ evaluate synthetic negatives có original outfit `>=3`.

Với mỗi candidate removal:

`delta_i = C(O \ x_i) - C(O)`

Metrics:
- LOO Top-1 Localization Accuracy
- LOO Hit@2
- breakdown theo outfit length


In [ ]:
loo_results = {}

for split in eval_splits:
    result = evaluate_loo_dataset(
        best_model,
        eval_datasets[split],
        device=DEVICE,
    )
    loo_results[split] = result

    print("\nLOO", split)
    print(json.dumps(result["metrics"], ensure_ascii=False, indent=2))
    print("by length:")
    for row in result["by_length"]:
        print(row)


## 11. Save evaluation artifacts


In [ ]:
EVAL_DIR = paths.scorer_ready_dir / "evaluation_min2_exp_v1"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "experiment": EXPERIMENT_TAG,
    "dataset_version": EXPERIMENT_DATASET_VERSION,
    "scorer_min_items": MIN_SCORER_ITEMS,
    "loo_min_original_items": LOO_MIN_ORIGINAL_ITEMS,
    "frozen_embedding_reference": {
        "source": str(REFERENCE_PATH),
        "dataset_version": DATA_REF["dataset_version"],
        "embedding_version": DATA_REF["embedding_version"],
        "cache_sha256": EXPECTED_CACHE_SHA256,
        "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    },
    "checkpoint": str(train_result["best_checkpoint"]),
    "checkpoint_epoch": int(ckpt["epoch"]),
    "scorer_metrics": scorer_metrics,
    "loo_metrics": {
        split: loo_results[split]["metrics"] for split in eval_splits
    },
    "loo_metrics_by_length": {
        split: loo_results[split]["by_length"] for split in eval_splits
    },
}

summary_path = EVAL_DIR / "evaluation_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
    f.write("\n")

for split in eval_splits:
    prediction_path = EVAL_DIR / f"loo_predictions_{split}.jsonl"
    with prediction_path.open("w", encoding="utf-8") as f:
        for row in loo_results[split]["predictions"]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("saved:", EVAL_DIR)


## Acceptance

Notebook chỉ đi tiếp tới training khi tất cả điều kiện sau đúng:

- cache `.pt` SHA-256 match frozen `data_v2_reference.json`;
- manifest SHA-256 match frozen reference;
- cache/manifest schema validation PASS;
- frozen cache đạt exact embedding coverage trên regenerated MIN2 positives;
- scorer-ready MIN2 validation `READY_TO_TRAIN`.

Nếu coverage không đạt, đó là một kết quả experiment quan trọng: cần explicit quyết định tạo **embedding artifact/version mới**, không được âm thầm mutate frozen V2 cache.
